# PLAN_B (FlowSDF variant) — Zero-shot transfer of frozen FlowSDF experts to RAM-H1200

Same experiment as `inference_new_dataset.ipynb`, but testing the **FlowSDF**
refinement experts instead of `cnnNoROI`. PLAN_B marked FlowSDF as a stretch
goal — "generative-from-noise on off-distribution embeddings is the most
likely to produce garbage and the hardest to interpret" — so treat this as
exploratory.

Key differences from the cnnNoROI notebook:
- FlowSDF conditions on (MedSAM embedding + coarse mask) at **128×128**, no
  explicit SDF input channel (cnnNoROI's expert takes embedding+mask+SDF at 64×64).
- Refinement is a flow-matching ODE sample (Euler integration through a
  ~160M-param UNet, `ODE_STEPS = 40` — see below), not a single forward pass
  through a ~371k-param CNN.
- Output is a signed distance field, thresholded to a binary mask
  (`sdf <= 0.03`), not a direct mask logit.

**`ODE_STEPS = 40`, not the script's own default of 4.** This repo's actual
PENGWIN inference job (`src/FlowSDF/jobs-slurmOutputs/2_infer_flowsdf.job`)
used `--ode-steps 40`, and this repo's own step-size ablation shows why that
matters: PENGWIN overall dice was ~0.78 at both 10 and 30 steps, but 0.908 at
40 steps (+0.19 dice vs. MedSAM baseline, 96.5% of fragments improved — see
`jobs-slurmOutputs/slurm_evaluate_flowsdf_stepsize{10,30}_*.out` vs.
`slurm_evaluate_flowsdf_24275752.out`). Step count is not a minor knob here;
under-stepping does not test this expert fairly.

That makes this ~10x more expensive per fragment than a 4-step run. Run with
the `MoE` conda env (`conda activate MoE`) as the kernel.

**Measured on this machine (MPS, warm): ~4.2s/fragment at 40 steps.** That puts
the smoke test (`SMOKE_LIMIT = 15`, ~450 fragments) at roughly **30 minutes**,
and a full test-split run (267 images, ~7900 fragments) at roughly **9 hours**
— start with the smoke test and only consider the full split as a deliberate,
long background run.


## 1. Setup

In [ ]:
import sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
WORKSPACE_ROOT = REPO_ROOT.parent
MEDSAM_REPO_ROOT = WORKSPACE_ROOT / "MedSAM"

for p in (REPO_ROOT / "src", REPO_ROOT / "src" / "gating_mechanism", REPO_ROOT / "src" / "FlowSDF", MEDSAM_REPO_ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from prepare_ramh1200_boxes import process_split
from run_medsam_with_pengwin_boxes import load_and_preprocess_image, run_medsam_with_boxes
from dataloader_utils import load_prediction_masks, resize_binary_nearest

from segment_anything import sam_model_registry

DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

DATASET_ROOT = REPO_ROOT / "RAM-H1200-v1_dataset" / "Segmentation"
BOXES_ROOT = REPO_ROOT / "data" / "ramh1200" / "bounding-boxes"
MEDSAM_PRED_ROOT = REPO_ROOT / "data" / "ramh1200" / "medsam-predictions"
FLOWSDF_PRED_ROOT = REPO_ROOT / "data" / "ramh1200" / "flowsdf-predictions"
CHECKPOINT_DIR = REPO_ROOT / "checkpoints" / "FlowSDF"
MEDSAM_CHECKPOINT = MEDSAM_REPO_ROOT / "medsam_vit_b.pth"

SPLIT = "test"
SMOKE_LIMIT = 15  # see Section 7 note: 40 ODE steps costs ~10x more per fragment than 4,
# so re-time the smoke test before deciding whether a full-split run is practical here.


## 2. Build boxes + GT masks for a small subset (adapter validation)

Same adapter as the cnnNoROI notebook — re-running it here is cheap and keeps this notebook self-contained.

In [ ]:
process_split(
    dataset_root=DATASET_ROOT,
    output_root=BOXES_ROOT,
    split=SPLIT,
    image_size=1024,
    limit=SMOKE_LIMIT,
)

records = [json.loads(l) for l in (BOXES_ROOT / SPLIT / "metadata.jsonl").open()]
print(f"{len(records)} records")
records[0]["sample_name"], records[0]["num_fragments"]


## 3. Visualize GT overlays — confirm masks + boxes line up with anatomy

In [ ]:
def show_gt_overlay(record, ax=None):
    image = np.array(Image.open(record["original_image_path"]).convert("L"))
    gt_masks = np.load(record["gt_masks_path"])["masks"]  # (N, H, W) original res

    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 9))
    ax.imshow(image, cmap="gray")

    overlay = np.zeros((*image.shape, 4))
    rng = np.random.default_rng(0)
    for i in range(gt_masks.shape[0]):
        color = rng.uniform(0.3, 1.0, size=3)
        m = gt_masks[i].astype(bool)
        overlay[m] = [*color, 0.45]
    ax.imshow(overlay)

    for frag in record["fragments"]:
        x1, y1, x2, y2 = frag["bbox_xyxy"]
        ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="cyan", linewidth=0.7))

    ax.set_title(f"{record['sample_name']} (N={record['num_fragments']})", fontsize=8)
    ax.axis("off")


fig, axes = plt.subplots(1, 3, figsize=(15, 9))
for ax, record in zip(axes, records[:3]):
    show_gt_overlay(record, ax=ax)
plt.tight_layout()
plt.show()


## 4. Stage-1 MedSAM inference

Identical to the cnnNoROI notebook — same preprocessing, same base checkpoint.
Both expert variants condition on the same MedSAM embeddings + coarse masks.

In [ ]:
model = sam_model_registry["vit_b"](checkpoint=str(MEDSAM_CHECKPOINT))
model = model.to(DEVICE)
model.eval()
print("loaded MedSAM base checkpoint:", MEDSAM_CHECKPOINT)


In [ ]:
binary_masks_dir = MEDSAM_PRED_ROOT / SPLIT / "binary_masks"
embeddings_dir = MEDSAM_PRED_ROOT / SPLIT / "embeddings"
binary_masks_dir.mkdir(parents=True, exist_ok=True)
embeddings_dir.mkdir(parents=True, exist_ok=True)

medsam_pred_records = []
for record in records:
    image = load_and_preprocess_image(record["original_image_path"], image_size=record["image_size"])
    boxes = np.load(record["box_path"]).astype(np.float32)

    binary_masks, embedding = run_medsam_with_boxes(model, image, boxes, DEVICE, threshold=0.5)

    sample_name = record["sample_name"]
    masks_path = binary_masks_dir / f"{sample_name}.npz"
    embedding_path = embeddings_dir / f"{sample_name}.npy"
    np.savez_compressed(masks_path, masks=binary_masks)
    np.save(embedding_path, embedding)

    medsam_pred_records.append({
        **record,
        "binary_masks_path": str(masks_path.resolve()),
        "embedding_path": str(embedding_path.resolve()),
    })

pred_metadata_path = MEDSAM_PRED_ROOT / SPLIT / "metadata.jsonl"
pred_metadata_path.parent.mkdir(parents=True, exist_ok=True)
with pred_metadata_path.open("w") as f:
    for r in medsam_pred_records:
        f.write(json.dumps(r) + "\n")

print(f"wrote {len(medsam_pred_records)} MedSAM prediction records to {pred_metadata_path}")


## 5. Sanity-check MedSAM baseline masks visually

In [ ]:
def show_pred_overlay(record, ax=None, title_prefix=""):
    image = np.array(Image.open(record["original_image_path"]).convert("L"))
    pred_masks = load_prediction_masks(record["binary_masks_path"])  # (N, 1024, 1024)

    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 9))
    ax.imshow(image, cmap="gray")

    h, w = image.shape
    overlay = np.zeros((*pred_masks.shape[1:], 4))
    rng = np.random.default_rng(1)
    for i in range(pred_masks.shape[0]):
        color = rng.uniform(0.3, 1.0, size=3)
        m = pred_masks[i].astype(bool)
        overlay[m] = [*color, 0.45]
    ax.imshow(overlay, extent=(0, w, h, 0))
    ax.set_title(f"{title_prefix}{record['sample_name']}", fontsize=8)
    ax.axis("off")


fig, axes = plt.subplots(1, 3, figsize=(15, 9))
for ax, record in zip(axes, medsam_pred_records[:3]):
    show_pred_overlay(record, ax=ax, title_prefix="MedSAM baseline: ")
plt.tight_layout()
plt.show()


## 6. Re-derive the gating threshold on RAM-H1200

Same procedure as the cnnNoROI notebook: median predicted-mask area on this
subset. Saved to a FlowSDF-specific CSV so this notebook doesn't clobber the
cnnNoROI notebook's gating file if both are open at once.

In [ ]:
areas = []
for record in medsam_pred_records:
    masks = load_prediction_masks(record["binary_masks_path"])
    for i in range(masks.shape[0]):
        areas.append(int(masks[i].sum()))

areas = np.array(areas)
ram_h1200_threshold = float(np.median(areas))
print(f"n fragments: {len(areas)}")
print(f"RAM-H1200 area threshold (median predicted-mask area, px on 1024 grid): {ram_h1200_threshold:.1f}")

def route_to_expert(area, threshold=ram_h1200_threshold):
    return "expert_small" if area <= threshold else "expert_large"

gated_rows = []
for record in medsam_pred_records:
    masks = load_prediction_masks(record["binary_masks_path"])
    for frag, mask in zip(record["fragments"], masks):
        area = int(mask.sum())
        gated_rows.append({
            "sample_name": record["sample_name"],
            "medsam_instance_id": frag["medsam_instance_id"],
            "category_name": frag["category_name"],
            "area": area,
            "expert": route_to_expert(area),
            "embedding_path": record["embedding_path"],
            "binary_masks_path": record["binary_masks_path"],
        })

gated_df = pd.DataFrame(gated_rows)
gating_dir = REPO_ROOT / "src" / "gating_mechanism"
gated_csv_path = gating_dir / f"gated_ramh1200_flowsdf_{SPLIT}_records.csv"
gated_df.to_csv(gated_csv_path, index=False)
print(f"saved {len(gated_df)} gated fragments -> {gated_csv_path}")
print(gated_df["expert"].value_counts().to_string())


## 7. Run frozen FlowSDF experts (no retraining)

Requires the PENGWIN-trained checkpoints at
`checkpoints/FlowSDF/expert_small_best.pth` and `expert_large_best.pth`
(already present in this repo). Each fragment costs one flow-matching ODE
sample — `ODE_STEPS = 40` Euler steps through a ~160M-param UNet (see the
intro cell for why 40, not the script's default of 4) — so this is much
slower than cnnNoROI, and slower than the earlier 4-step run of this same
notebook. Measured on this machine: ~4.2s/fragment on MPS (warm) at 40 steps — about
30 min for this notebook's N=15 smoke test. Budget accordingly before
switching `SMOKE_LIMIT` to `None` (~9 hours for the full test split).


In [ ]:
from models import unet_segdiff

FLOWSDF_IMG_SIZE = 128
SIGMA_MIN = 1e-5
ODE_STEPS = 40  # matches src/FlowSDF/jobs-slurmOutputs/2_infer_flowsdf.job, the job that produced this repo's reported PENGWIN numbers.
# infer_flowsdf_moe.py's own --ode-steps default is 4, which is NOT what was used to
# evaluate on PENGWIN. Per this repo's own step-size ablation
# (jobs-slurmOutputs/slurm_evaluate_flowsdf_stepsize{10,30}_*.out vs the 40-step run),
# step count is not a minor knob: PENGWIN overall dice was ~0.78 at 10 and 30 steps, but
# 0.908 at 40 steps (+0.19 dice / 96.5% of fragments improved vs MedSAM baseline). Fewer
# than 40 steps is not a faithful test of this expert.
N_EVAL = 1
SDF_BINARY_THRESHOLD = 0.03


def build_flowsdf_model(img_cond_channels, device):
    return unet_segdiff.UNetModel(
        in_channels=1,
        model_channels=128,
        out_channels=1,
        num_res_blocks=3,
        attention_resolutions=(16, 8),
        dropout=0,
        channel_mult=(1, 1, 2, 2, 4, 4),
        conv_resample=True,
        dims=2,
        num_classes=None,
        use_checkpoint=False,
        num_heads=1,
        num_heads_upsample=-1,
        use_scale_shift_norm=False,
        rrdb_blocks=12,
        img_cond_channels=img_cond_channels,
    ).to(device)


def load_flowsdf_expert(checkpoint_path, device):
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=True)
    img_cond_channels = int(ckpt.get("img_cond_channels", 257))
    model = build_flowsdf_model(img_cond_channels, device)
    model.load_state_dict(ckpt["ema_state"])
    model.eval()
    n_params = sum(p.numel() for p in model.parameters())
    print(f"loaded {checkpoint_path.name}: img_cond_channels={img_cond_channels}, params={n_params:,}")
    return model


flowsdf_experts = {}
for expert_id in ("expert_small", "expert_large"):
    ckpt_path = CHECKPOINT_DIR / f"{expert_id}_best.pth"
    if not ckpt_path.exists():
        raise FileNotFoundError(
            f"Checkpoint not found: {ckpt_path}\n"
            f"Copy the PENGWIN-trained FlowSDF checkpoints into {CHECKPOINT_DIR} and re-run this cell."
        )
    flowsdf_experts[expert_id] = load_flowsdf_expert(ckpt_path, DEVICE)


In [ ]:
import torch.nn.functional as F


def prepare_flowsdf_conditioning(embedding, binary_mask, img_size, device):
    embedding = embedding.to(device=device, dtype=torch.float32)
    embedding_resized = F.interpolate(
        embedding.unsqueeze(0), size=(img_size, img_size), mode="bilinear", align_corners=False
    )
    mask = torch.from_numpy(binary_mask.astype(np.float32)).to(device).unsqueeze(0).unsqueeze(0)
    mask_resized = F.interpolate(mask, size=(img_size, img_size), mode="nearest")
    mask_resized = (mask_resized > 0.5).float()
    return torch.cat([embedding_resized, mask_resized], dim=1)


@torch.no_grad()
def sample_flowsdf_sdf(model, img_cond, sigma_min, ode_steps, n_eval):
    samples = []
    device = img_cond.device
    _, _, height, width = img_cond.shape

    for _ in range(n_eval):
        m0 = torch.randn((1, 1, height, width), device=device)

        def func_conditional(t, m):
            t_curr = torch.ones(m.shape[0], device=m.device) * t
            m_ipt = (1 - (1 - sigma_min) * t) * m0 + t * m
            return model(m_ipt, t_curr.reshape(m.shape[0], -1), img_cond=img_cond)

        times = torch.linspace(0, 1, ode_steps, device=device)
        m = m0
        for i in range(len(times) - 1):
            dt = times[i + 1] - times[i]
            m = m + dt * func_conditional(times[i], m)
        samples.append(m)

    if len(samples) == 1:
        return samples[0]
    return torch.stack(samples, dim=0).mean(dim=0)


@torch.no_grad()
def refine_fragment_flowsdf(embedding, binary_mask, model, device):
    img_cond = prepare_flowsdf_conditioning(embedding, binary_mask, FLOWSDF_IMG_SIZE, device)
    sdf = sample_flowsdf_sdf(model, img_cond, SIGMA_MIN, ODE_STEPS, N_EVAL)
    mask_small = (sdf <= SDF_BINARY_THRESHOLD).float()
    mask_up = F.interpolate(mask_small, size=(1024, 1024), mode="nearest")
    return mask_up[0, 0].cpu().numpy().astype(np.uint8)


flowsdf_masks_dir = FLOWSDF_PRED_ROOT / SPLIT / "binary_masks"
flowsdf_masks_dir.mkdir(parents=True, exist_ok=True)

from tqdm import tqdm

for sample_name in tqdm(gated_df["sample_name"].unique(), desc="FlowSDF inference"):
    rows = gated_df[gated_df["sample_name"] == sample_name].sort_values("medsam_instance_id").reset_index(drop=True)

    embedding = torch.from_numpy(np.load(rows.iloc[0]["embedding_path"])).float().to(DEVICE)
    all_masks = load_prediction_masks(rows.iloc[0]["binary_masks_path"])

    refined = []
    for _, row in rows.iterrows():
        idx = int(row["medsam_instance_id"]) - 1
        binary_mask = all_masks[idx]
        expert_model = flowsdf_experts[row["expert"]]
        refined.append(refine_fragment_flowsdf(embedding, binary_mask, expert_model, DEVICE))

    out_masks = np.stack(refined)
    np.savez_compressed(flowsdf_masks_dir / f"{sample_name}.npz", masks=out_masks)

print(f"refined masks written to {flowsdf_masks_dir}")


## 8. Evaluate: MedSAM baseline vs. MedSAM + frozen FlowSDF

Reuses the exact dice/iou/hd95/assd implementations from
`evaluation/evaluate_medsam_pengwin.py`.

In [ ]:
sys.path.insert(0, str(REPO_ROOT))
from evaluation.evaluate_medsam_pengwin import dice_score, iou_score, boundary_metrics

def evaluate_predictions(pred_masks_dir, gated_df, skip_boundary=False):
    rows = []
    for record in medsam_pred_records:
        sample_name = record["sample_name"]
        gt_masks = np.load(record["gt_masks_path"])["masks"]  # (N, H, W) original res
        pred_path = pred_masks_dir / f"{sample_name}.npz"
        pred_masks = load_prediction_masks(pred_path)  # (N, 1024, 1024)

        frag_expert = {
            int(r["medsam_instance_id"]): r["expert"]
            for _, r in gated_df[gated_df["sample_name"] == sample_name].iterrows()
        }

        for i, frag in enumerate(record["fragments"]):
            gt = gt_masks[i]
            pred = resize_binary_nearest(pred_masks[i], gt.shape)

            dice = dice_score(pred, gt)
            iou = iou_score(pred, gt)
            hd95, assd = (np.nan, np.nan) if skip_boundary else boundary_metrics(pred, gt)

            rows.append({
                "sample_name": sample_name,
                "category_name": frag["category_name"],
                "size_group": frag_expert.get(frag["medsam_instance_id"], "unknown"),
                "dice": dice, "iou": iou, "hd95": hd95, "assd": assd,
            })
    return pd.DataFrame(rows)


baseline_df = evaluate_predictions(MEDSAM_PRED_ROOT / SPLIT / "binary_masks", gated_df)
flowsdf_df = evaluate_predictions(flowsdf_masks_dir, gated_df)

print("MedSAM baseline (overall):")
print(baseline_df[["dice", "iou", "hd95", "assd"]].mean())
print("\nMedSAM + frozen FlowSDF (overall):")
print(flowsdf_df[["dice", "iou", "hd95", "assd"]].mean())

print("\nMedSAM baseline by size group:")
print(baseline_df.groupby("size_group")[["dice", "iou", "hd95", "assd"]].mean())
print("\nMedSAM + frozen FlowSDF by size group:")
print(flowsdf_df.groupby("size_group")[["dice", "iou", "hd95", "assd"]].mean())


## 9. Where are we off (and on)? GT vs. MedSAM baseline vs. frozen-FlowSDF-refined

Worst-regression, median/typical, and best-improvement cases side by side,
so both the failure mode and any genuine gain are visible rather than
summarized only in aggregate metrics.

In [ ]:
delta_rows = []
for record in medsam_pred_records:
    sample_name = record["sample_name"]
    gt_masks = np.load(record["gt_masks_path"])["masks"]
    baseline_masks = load_prediction_masks(MEDSAM_PRED_ROOT / SPLIT / "binary_masks" / f"{sample_name}.npz")
    flowsdf_masks = load_prediction_masks(flowsdf_masks_dir / f"{sample_name}.npz")

    base_dices, flow_dices = [], []
    for i in range(gt_masks.shape[0]):
        gt = gt_masks[i]
        base_dices.append(dice_score(resize_binary_nearest(baseline_masks[i], gt.shape), gt))
        flow_dices.append(dice_score(resize_binary_nearest(flowsdf_masks[i], gt.shape), gt))

    delta_rows.append({
        "sample_name": sample_name,
        "base_dice": np.mean(base_dices),
        "flowsdf_dice": np.mean(flow_dices),
        "delta": np.mean(flow_dices) - np.mean(base_dices),
    })

delta_df = pd.DataFrame(delta_rows).sort_values("delta").reset_index(drop=True)
worst_case = delta_df.iloc[0]["sample_name"]
median_case = delta_df.iloc[len(delta_df) // 2]["sample_name"]
best_case = delta_df.iloc[-1]["sample_name"]
print(f"worst regression: {worst_case} (delta={delta_df.iloc[0]['delta']:.3f})")
print(f"median/typical:   {median_case} (delta={delta_df.iloc[len(delta_df)//2]['delta']:.3f})")
print(f"best improvement: {best_case} (delta={delta_df.iloc[-1]['delta']:.3f})")

records_by_name = {r["sample_name"]: r for r in medsam_pred_records}

def colorize(masks, rng_seed=0):
    overlay = np.zeros((*masks.shape[1:], 4))
    rng = np.random.default_rng(rng_seed)
    for i in range(masks.shape[0]):
        color = rng.uniform(0.3, 1.0, size=3)
        m = masks[i].astype(bool)
        overlay[m] = [*color, 0.45]
    return overlay

fig, axes = plt.subplots(3, 3, figsize=(15, 18))
for row, (sample_name, row_label) in enumerate([
    (worst_case, "worst"),
    (median_case, "median"),
    (best_case, "best"),
]):
    record = records_by_name[sample_name]
    image = np.array(Image.open(record["original_image_path"]).convert("L"))
    h, w = image.shape

    gt_masks = np.load(record["gt_masks_path"])["masks"]
    baseline_masks = load_prediction_masks(MEDSAM_PRED_ROOT / SPLIT / "binary_masks" / f"{sample_name}.npz")
    flowsdf_masks = load_prediction_masks(flowsdf_masks_dir / f"{sample_name}.npz")

    for col, (masks, title) in enumerate([
        (gt_masks, "GT"),
        (baseline_masks, "MedSAM baseline"),
        (flowsdf_masks, "MedSAM + frozen FlowSDF"),
    ]):
        ax = axes[row, col]
        ax.imshow(image, cmap="gray", extent=(0, w, h, 0))
        ax.imshow(colorize(masks), extent=(0, w, h, 0))
        ax.set_title(f"{title}\n{row_label}: {sample_name}" if col == 0 else title, fontsize=8)
        ax.axis("off")

plt.tight_layout()
plt.show()


### Export the individual panels (no titles/labels) for a custom figure

Saves each of the 9 panels above (X-ray + colorized mask overlay, nothing
else — no title, no axis, no matplotlib text) as its own PNG, so captions,
labels, and font sizes can be added afterward outside this notebook.

In [ ]:
FIGURES_DIR = REPO_ROOT / "figures" / "ramh1200_flowsdf" / SPLIT
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

panel_manifest = []
for sample_name, row_label in [
    (worst_case, "worst"),
    (median_case, "median"),
    (best_case, "best"),
]:
    record = records_by_name[sample_name]
    image = np.array(Image.open(record["original_image_path"]).convert("L"))
    h, w = image.shape

    gt_masks = np.load(record["gt_masks_path"])["masks"]
    baseline_masks = load_prediction_masks(MEDSAM_PRED_ROOT / SPLIT / "binary_masks" / f"{sample_name}.npz")
    flowsdf_masks = load_prediction_masks(flowsdf_masks_dir / f"{sample_name}.npz")

    for col_label, masks in [
        ("gt", gt_masks),
        ("medsam_baseline", baseline_masks),
        ("flowsdf_refined", flowsdf_masks),
    ]:
        panel_fig, panel_ax = plt.subplots(figsize=(w / 150, h / 150), dpi=150)
        panel_ax.imshow(image, cmap="gray", extent=(0, w, h, 0))
        panel_ax.imshow(colorize(masks), extent=(0, w, h, 0))
        panel_ax.axis("off")
        panel_ax.set_position([0, 0, 1, 1])  # fill the whole figure, no margins

        out_path = FIGURES_DIR / f"{row_label}_{sample_name}_{col_label}.png"
        panel_fig.savefig(out_path, dpi=150, bbox_inches="tight", pad_inches=0)
        plt.close(panel_fig)

        panel_manifest.append({
            "case": row_label, "sample_name": sample_name, "panel": col_label, "path": str(out_path),
        })

manifest_df = pd.DataFrame(panel_manifest)
manifest_df.to_csv(FIGURES_DIR / "manifest.csv", index=False)
print(f"saved {len(panel_manifest)} panels to {FIGURES_DIR}")
manifest_df


## 11. Is RAM-H1200's "large" actually large? (PENGWIN-relative sizing)

The gating threshold in Section 6 is RAM-H1200-*relative* — it's just this
dataset's own median fragment area, so "large" only means "above the RAM-H1200
median," regardless of what that area actually is in absolute pixels.

PENGWIN's own fragment-area analysis
(`data_distribution_results/data_analysis_gt_with_medsam/threshold_suggestions.csv`)
tells us what "small" meant when these experts were trained: their 5402px
threshold is exactly PENGWIN's **25th percentile** fragment area — most PENGWIN
fragments are much bigger than that. If RAM-H1200 bones sit low on PENGWIN's
absolute scale even when they're "large" *relative to other RAM-H1200 bones*,
that's a concrete, checkable reason `expert_large` might be regressing: it was
never trained on anything this small, regardless of which RAM-H1200 expert it
got routed to.

In [ ]:
pengwin_pct_path = REPO_ROOT / "data_distribution_results" / "data_analysis_gt_with_medsam" / "threshold_suggestions.csv"
pengwin_pct = pd.read_csv(pengwin_pct_path)
print("PENGWIN fragment area percentiles (px, from GT+MedSAM analysis):")
print(pengwin_pct.to_string(index=False))

PENGWIN_SMALL_THRESHOLD = 5402.0   # the threshold actually used to train expert_small/expert_large
PENGWIN_MEDIAN = float(pengwin_pct.loc[pengwin_pct["metric"] == "area", "p50"].iloc[0])

print(f"\nPENGWIN's small/large split point: {PENGWIN_SMALL_THRESHOLD:.0f}px (their p25)")
print(f"PENGWIN median fragment area:        {PENGWIN_MEDIAN:.0f}px")
print(f"RAM-H1200 median fragment area:      {ram_h1200_threshold:.0f}px  <- this notebook's current split point")


In [ ]:
pct_below_pengwin_small = (gated_df["area"] <= PENGWIN_SMALL_THRESHOLD).mean()
pct_below_pengwin_median = (gated_df["area"] <= PENGWIN_MEDIAN).mean()

print(f"RAM-H1200 fragments below PENGWIN's small-expert threshold (5402px): {pct_below_pengwin_small:.1%}")
print(f"RAM-H1200 fragments below PENGWIN's own median fragment size:        {pct_below_pengwin_median:.1%}")

large_group = gated_df[gated_df["expert"] == "expert_large"]
flipped = large_group[large_group["area"] <= PENGWIN_SMALL_THRESHOLD]
print(f"\nOf RAM-H1200's current expert_large fragments (n={len(large_group)}), "
      f"{len(flipped)} ({len(flipped) / len(large_group):.1%}) are still within "
      f"PENGWIN's *own* small-expert range once you use PENGWIN's absolute threshold "
      f"instead of RAM-H1200's relative median.")
print(f"expert_large median area: {large_group['area'].median():.0f}px "
      f"(for reference, PENGWIN's own median fragment: {PENGWIN_MEDIAN:.0f}px)")


### Re-route the flipped fragments through `expert_small` and re-evaluate

Only the fragments identified above change expert assignment — everything
else keeps its already-computed refined mask, so this only re-runs FlowSDF on
the subset that actually flips (this can still be a few thousand fragments;
it doesn't need to be all of `expert_large`).

In [ ]:
pengwin_masks_dir = FLOWSDF_PRED_ROOT / SPLIT / "binary_masks_pengwin_convention"
pengwin_masks_dir.mkdir(parents=True, exist_ok=True)

gated_df_pengwin = gated_df.copy()
gated_df_pengwin["expert"] = np.where(
    gated_df_pengwin["area"] <= PENGWIN_SMALL_THRESHOLD, "expert_small", "expert_large"
)

changed = (gated_df_pengwin["expert"] != gated_df["expert"]).sum()
print(f"{changed} / {len(gated_df)} fragments change expert assignment "
      f"under PENGWIN-absolute routing ({changed / len(gated_df):.1%})")
print(gated_df_pengwin["expert"].value_counts().to_string())

for sample_name in tqdm(gated_df_pengwin["sample_name"].unique(), desc="Re-routing + FlowSDF (flipped fragments only)"):
    old_rows = gated_df[gated_df["sample_name"] == sample_name].sort_values("medsam_instance_id").reset_index(drop=True)
    new_rows = gated_df_pengwin[gated_df_pengwin["sample_name"] == sample_name].sort_values("medsam_instance_id").reset_index(drop=True)

    # start from the already-computed refined masks for this image
    out_masks = load_prediction_masks(flowsdf_masks_dir / f"{sample_name}.npz").copy()

    embedding = None
    all_medsam_masks = None
    for (_, old_row), (_, new_row) in zip(old_rows.iterrows(), new_rows.iterrows()):
        if old_row["expert"] == new_row["expert"]:
            continue  # unchanged, keep the already-computed refined mask

        if embedding is None:
            embedding = torch.from_numpy(np.load(new_row["embedding_path"])).float().to(DEVICE)
            all_medsam_masks = load_prediction_masks(new_row["binary_masks_path"])

        idx = int(new_row["medsam_instance_id"]) - 1
        expert_model = flowsdf_experts[new_row["expert"]]
        out_masks[idx] = refine_fragment_flowsdf(embedding, all_medsam_masks[idx], expert_model, DEVICE)

    np.savez_compressed(pengwin_masks_dir / f"{sample_name}.npz", masks=out_masks)

print(f"PENGWIN-convention refined masks written to {pengwin_masks_dir}")


In [ ]:
pengwin_routed_df = evaluate_predictions(pengwin_masks_dir, gated_df_pengwin)

print("MedSAM baseline (overall):")
print(baseline_df[["dice", "iou", "hd95", "assd"]].mean())
print("\nMedSAM + frozen FlowSDF, RAM-H1200-relative routing (overall):")
print(flowsdf_df[["dice", "iou", "hd95", "assd"]].mean())
print("\nMedSAM + frozen FlowSDF, PENGWIN-absolute routing (overall):")
print(pengwin_routed_df[["dice", "iou", "hd95", "assd"]].mean())

# Isolate the effect: for exactly the fragments that flip from expert_large to
# expert_small, compare baseline vs old (expert_large) refined vs new (expert_small)
# refined, fragment by fragment. This is the cleanest read on whether re-routing helps,
# since it holds the fragment set fixed and only changes which expert refined it.
flipped_keys = set(
    map(tuple, gated_df_pengwin.loc[gated_df_pengwin["expert"] != gated_df["expert"], ["sample_name", "medsam_instance_id"]].values)
)
print(f"\nIsolating the {len(flipped_keys)} flipped fragments (expert_large -> expert_small under PENGWIN routing):")

flipped_rows = []
for sample_name in {k[0] for k in flipped_keys}:
    record = records_by_name[sample_name]
    gt_masks = np.load(record["gt_masks_path"])["masks"]
    baseline_masks = load_prediction_masks(MEDSAM_PRED_ROOT / SPLIT / "binary_masks" / f"{sample_name}.npz")
    old_masks = load_prediction_masks(flowsdf_masks_dir / f"{sample_name}.npz")
    new_masks = load_prediction_masks(pengwin_masks_dir / f"{sample_name}.npz")

    for frag in record["fragments"]:
        key = (sample_name, frag["medsam_instance_id"])
        if key not in flipped_keys:
            continue
        idx = frag["medsam_instance_id"] - 1
        gt = gt_masks[idx]
        flipped_rows.append({
            "sample_name": sample_name,
            "baseline_dice": dice_score(resize_binary_nearest(baseline_masks[idx], gt.shape), gt),
            "expert_large_dice": dice_score(resize_binary_nearest(old_masks[idx], gt.shape), gt),
            "expert_small_dice": dice_score(resize_binary_nearest(new_masks[idx], gt.shape), gt),
        })

flipped_df = pd.DataFrame(flipped_rows)
print(flipped_df[["baseline_dice", "expert_large_dice", "expert_small_dice"]].mean())


Interpreting this: if PENGWIN-absolute routing recovers some of the
`expert_large` regression (without hurting `expert_small`), that supports the
"still too small for this expert" story — the fix isn't a better `expert_large`,
it's recognizing these bones never leave the small-object regime PENGWIN
trained on. If it doesn't help, the regression is more likely about anatomy
(pelvis fragment shapes vs. hand bone shapes) than about absolute scale, and
this reframing wouldn't be honest to lean on.

## 12. Next steps
- If the smoke test (N=15) looks reasonable, consider `N_EVAL` (average
  multiple ODE trajectories) or `ODE_STEPS` (finer integration) before
  scaling up — FlowSDF is stochastic per-sample, unlike the deterministic
  cnnNoROI forward pass, so a single trajectory may be noisy.
- Scaling to the full `test` split (267 images, ~8000 fragments, each costing
  an ODE sample) will take much longer than the cnnNoROI run — time the
  smoke test first and extrapolate before committing to a full run.
- Compare directly against the cnnNoROI results in `inference_new_dataset.ipynb`
  once both are available.